# BioViz Studio — benchmark analysis

Reproduces the latency and throughput figures in the SoftwareX paper from the benchmark CSVs exported by the app's **Evaluation Experiment** mode.

Place the CSVs under `eval_data/` as listed in `eval_data/README.md`, or set the `BIOVIZ_EVAL_DATA` environment variable to another folder.

In [ ]:
import os
from pathlib import Path

# Folder holding the benchmark CSVs (relative to this notebook by default)
DATA_DIR = Path(os.environ.get('BIOVIZ_EVAL_DATA', 'eval_data'))
assert DATA_DIR.exists(), f'Benchmark data folder not found: {DATA_DIR.resolve()}'

In [ ]:
import pandas as pd
import numpy as np 
import plotly.express as px

In [ ]:
df = pd.read_csv(f'{DATA_DIR}/MIT_BIH_Stress_Noise_Trace1.csv')


In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
import pandas as pd
import plotly.express as px


# 2. Create a unique numerical mapping for each file
unique_files = sorted(df['file_name'].unique())
file_to_id = {name: i + 1 for i, name in enumerate(unique_files)}
df['file_id'] = df['file_name'].map(file_to_id)

# 3. Aggregate data to get the mean of the 5 samples per file
df_agg = df.groupby(['file_id', 'file_name'])['plot_gen_time_ms'].mean().reset_index()

# 4. Sort by file_id in ascending order (1, 2, 3...)
df_agg = df_agg.sort_values(by='file_id', ascending=True)

# Convert ID to string to ensure discrete categorical labeling
df_agg['file_id'] = df_agg['file_id'].astype(str)

# 5. Create the Plotly Bar Chart
fig = px.bar(
    df_agg,
    x='file_id',
    y='plot_gen_time_ms',
    title='Plot Generation Time per File (Sorted by ID)',
    labels={'file_id': 'Unique File ID', 'plot_gen_time_ms': 'Plot Generation Time (ms)'},
    template='plotly_white'
)

# 6. Apply Research Paper Formatting with Increased Font Sizes
fig.update_traces(
    marker_color='#4E79A7',    # Uniform professional blue
    marker_line_color='black', # Defined borders
    marker_line_width=1.5,
    customdata=df_agg['file_name'],
    hovertemplate="<b>File ID: %{x}</b><br>Original Name: %{customdata}<br>Avg Time: %{y:.2f} ms<extra></extra>"
)

fig.update_layout(
    font_family="Arial",
    font_color="black",
    title_font_size=26,         # Main Title Size
    xaxis=dict(
        title="File Identifier (ID)",
        title_font_size=22,     # Increased X-axis label size
        tickfont_size=22,       # Increased X-axis tick size
        tickangle=0,
        showgrid=False,
        type='category'
    ),
    yaxis=dict(
        title="Plot Generation Time (ms)",
        title_font_size=22,     # Increased Y-axis label size
        tickfont_size=22,       # Increased Y-axis tick size
        showgrid=True,
        gridcolor='LightGray'
    ),
    width=1000,
    height=600,
    margin=dict(l=80, r=50, t=100, b=100)
)

fig.show()

In [ ]:
df

In [ ]:



# 2. Create a unique numerical mapping for each file
unique_files = sorted(df['file_name'].unique())
file_to_id = {name: i + 1 for i, name in enumerate(unique_files)}
df['file_id'] = df['file_name'].map(file_to_id)

# 3. Aggregate data to get the mean of the 5 samples per file
df_agg = df.groupby(['file_id', 'file_name'])['throughput_ksps'].mean().reset_index()

# 4. Sort by file_id in ascending order (1, 2, 3...)
df_agg = df_agg.sort_values(by='file_id', ascending=True)

# Convert ID to string to ensure discrete categorical labeling
df_agg['file_id'] = df_agg['file_id'].astype(str)

# 5. Create the Plotly Bar Chart
fig = px.bar(
    df_agg,
    x='file_id',
    y='throughput_ksps',
    title='Throughput per File (Sorted by ID)',
    labels={'file_id': 'Unique File ID', 'throughput_ksps': 'Throughput (ksps)'},
    template='plotly_white'
)

# 6. Apply Research Paper Formatting with Increased Font Sizes
fig.update_traces(
    marker_color='#4E79A7',    # Uniform professional blue
    marker_line_color='black', # Defined borders
    marker_line_width=1.5,
    customdata=df_agg['file_name'],
    hovertemplate="<b>File ID: %{x}</b><br>Original Name: %{customdata}<br>Avg Time: %{y:.2f} ms<extra></extra>"
)

fig.update_layout(
    font_family="Arial",
    font_color="black",
    title_font_size=26,         # Main Title Size
    xaxis=dict(
        title="File Identifier (ID)",
        title_font_size=22,     # Increased X-axis label size
        tickfont_size=22,       # Increased X-axis tick size
        tickangle=0,
        showgrid=False,
        type='category'
    ),
    yaxis=dict(
        title="Throughput (ksps)",
        title_font_size=22,     # Increased Y-axis label size
        tickfont_size=22,       # Increased Y-axis tick size
        showgrid=True,
        gridcolor='LightGray'
    ),
    width=1000,
    height=600,
    margin=dict(l=80, r=50, t=100, b=100)
)

fig.show()

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================
# 1. Load the datasets
# =========================
file_trace1 = f'{DATA_DIR}/MIT_BIH_LongTermData/trace1_MITBIH_LongTerm_benchmark_results_EVAL_BATCH_tmp57v9l9qd_20260120_171527_21ba.csv'
file_trace2 = f'{DATA_DIR}/MIT_BIH_LongTermData/trace2_MITBIH_LongTerm_benchmark_results_EVAL_BATCH_tmp57v9l9qd_20260120_171643_0cb4.csv'

df1 = pd.read_csv(file_trace1)
df2 = pd.read_csv(file_trace2)

# =========================
# 2. Combine & map serial numbers
# =========================
df_combined = pd.concat([df1, df2], ignore_index=True)

unique_files = sorted(df_combined['file_name'].unique())
file_map = {name: i+1 for i, name in enumerate(unique_files)}
df_combined['serial_number'] = df_combined['file_name'].map(file_map)

# =========================
# 3. Aggregate averages
# =========================
df_agg = (
    df_combined
    .groupby(['serial_number','active_trace_count'])
    [['plot_gen_time_ms','throughput_ksps']]
    .mean()
    .reset_index()
)

trace1 = df_agg[df_agg['active_trace_count']==1].sort_values('serial_number')
trace2 = df_agg[df_agg['active_trace_count']==2].sort_values('serial_number')

# =========================
# 4. Create Subplots
# =========================
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=[
        "(a) Average Plot Generation Time",
        "(b) Average Throughput"
    ],
    vertical_spacing=0.18
)

# =========================
# Publication-friendly colors
# (colorblind & grayscale safe)
# =========================
color_trace1 = "red"   # dark gray
color_trace2 = "blue"   # light gray

bar_width = 0.32  # thinner bars for clarity

# ---- Plot Generation Time ----
fig.add_trace(go.Bar(
    x=trace1['serial_number'],
    y=trace1['plot_gen_time_ms'],
    name='1 Active Trace',
    marker_color=color_trace1,
    width=bar_width
), row=1, col=1)

fig.add_trace(go.Bar(
    x=trace2['serial_number'],
    y=trace2['plot_gen_time_ms'],
    name='2 Active Traces',
    marker_color=color_trace2,
    width=bar_width
), row=1, col=1)

# ---- Throughput ----
fig.add_trace(go.Bar(
    x=trace1['serial_number'],
    y=trace1['throughput_ksps'],
    marker_color=color_trace1,
    width=bar_width,
    showlegend=False
), row=2, col=1)

fig.add_trace(go.Bar(
    x=trace2['serial_number'],
    y=trace2['throughput_ksps'],
    marker_color=color_trace2,
    width=bar_width,
    showlegend=False
), row=2, col=1)

# =========================
# 5. IEEE/ACM Layout Styling
# =========================
fig.update_layout(
    barmode='group',

    # spacing improvements
    bargap=0.38,        # space between file groups
    bargroupgap=0.10,   # space between bars inside group

    template="plotly_white",

    # ideal size for double column figures
    width=720,
    height=820,

    # clean legend placement
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=16)
    ),

    margin=dict(l=80, r=30, t=90, b=70)
)

# =========================
# 6. Axis Styling (publication clarity)
# =========================
axis_font = dict(size=20)

for row in [1,2]:
    fig.update_xaxes(
        title="File Serial Number",
        tickmode='linear',
        showline=True,
        linewidth=1,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickfont=axis_font,
        title_font=axis_font,
        row=row, col=1
    )

fig.update_yaxes(
    title="Time (ms)",
    showline=True,
    linewidth=1,
    linecolor='black',
    mirror=True,
    gridcolor='rgba(0,0,0,0.15)',
    tickfont=axis_font,
    title_font=axis_font,
    row=1, col=1
)

fig.update_yaxes(
    title="Throughput (ksps)",
    showline=True,
    linewidth=1,
    linecolor='black',
    mirror=True,
    gridcolor='rgba(0,0,0,0.15)',
    tickfont=axis_font,
    title_font=axis_font,
    row=2, col=1
)

# subplot title font
fig.update_annotations(font_size=18)

fig.show()

# =========================
# 7. (Optional) High-Resolution Export
# =========================
# Install kaleido first: pip install kaleido
# fig.write_image("figure7.pdf")   # vector (best for papers)
# fig.write_image("figure7.png", scale=3)


## MIT BIH Stress Noise Local vs Container

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------------------------------------------------------
# 🎨 COLOR CONFIGURATION
# ---------------------------------------------------------
COLOR_LOCAL = '#EF553B'      # Red/Orange
COLOR_DOCKER = '#636EFA'     # Blue/Purple

# ---------------------------------------------------------
# 1. Load the Datasets
# ---------------------------------------------------------
files = {
    "Local_1": f"{DATA_DIR}/MIT_BIH_Stress_Noise_Trace2_LocalDesktop/MIT_BIH_Stress_Trace1_LocalDesktop.csv",
    "Local_2": f"{DATA_DIR}/MIT_BIH_Stress_Noise_Trace2_LocalDesktop/MIT_BIH_Stress_Trace2_LocalDesktop.csv",
    "Docker_1": f"{DATA_DIR}/MIT_BIH_Stress_Noise_Trace2_LocalDesktop_Container/MIT_BIH_Stress_Noise_Trace1.csv",
    "Docker_2": f"{DATA_DIR}/MIT_BIH_Stress_Noise_Trace2_LocalDesktop_Container/MITBIH_Stress_Noise_Trace2.csv"
}

try:
    df_local_1 = pd.read_csv(files["Local_1"])
    df_local_2 = pd.read_csv(files["Local_2"])
    df_docker_1 = pd.read_csv(files["Docker_1"])
    df_docker_2 = pd.read_csv(files["Docker_2"])
except FileNotFoundError as e:
    print(f"Error: {e}. Please check your file paths.")
    exit()

# ---------------------------------------------------------
# 2. Preprocess & Merge
# ---------------------------------------------------------
df_local_1['Environment'] = 'Local (Mac Pro)'
df_local_1['Traces'] = '1 Trace'
df_local_2['Environment'] = 'Local (Mac Pro)'
df_local_2['Traces'] = '2 Traces'

df_docker_1['Environment'] = 'Docker Container'
df_docker_1['Traces'] = '1 Trace'
df_docker_2['Environment'] = 'Docker Container'
df_docker_2['Traces'] = '2 Traces'

df_all = pd.concat([df_local_1, df_local_2, df_docker_1, df_docker_2], ignore_index=True)

# ---------------------------------------------------------
# 3. Create Plots (Vertical Layout)
# ---------------------------------------------------------
colors = {'Local (Mac Pro)': COLOR_LOCAL, 'Docker Container': COLOR_DOCKER}

# Font size settings for two-column paper visibility
font_size_axis = 18
font_size_title = 20
font_size_legend = 18

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("(a) <b>Execution Latency</b> (Lower is Better)", 
                    "(b) <b>System Throughput</b> (Higher is Better)"),
    vertical_spacing=0.12
)

# Plot 1: Execution Time (Row 1)
# Using sorted unique values to ensure alignment in 'group' mode
for env in ['Local (Mac Pro)', 'Docker Container']:
    subset = df_all[df_all['Environment'] == env]
    fig.add_trace(
        go.Box(
            x=subset['Traces'],
            y=subset['execution_time_ms'],
            name=env,
            legendgroup=env,
            marker_color=colors[env],
            boxpoints='outliers',
            line_width=1.5,
            offsetgroup=env # Ensures distinct groups for centering
        ),
        row=1, col=1
    )

# Plot 2: Throughput (Row 2)
for env in ['Local (Mac Pro)', 'Docker Container']:
    subset = df_all[df_all['Environment'] == env]
    fig.add_trace(
        go.Box(
            x=subset['Traces'],
            y=subset['throughput_ksps'],
            name=env,
            legendgroup=env,
            showlegend=False,
            marker_color=colors[env],
            boxpoints='outliers',
            line_width=1.5,
            offsetgroup=env # Ensures distinct groups for centering
        ),
        row=2, col=1
    )

# ---------------------------------------------------------
# 4. Update Layout & Axis Styling
# ---------------------------------------------------------
fig.update_layout(
    title_text="Performance Benchmarking: Native vs. Containerized (MIT BIH)",
    title_x=0.5,
    title_font_size=22,
    boxmode='group', # This is critical for centering labels between grouped boxes
    height=900,
    width=800,
    template="plotly_white",
    
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        font=dict(size=font_size_legend)
    ),
    margin=dict(l=80, r=40, t=120, b=80)
)

# Row 1 Axis Styling
fig.update_yaxes(
    title_text="Time (ms)", 
    title_font=dict(size=font_size_axis),
    tickfont=dict(size=font_size_axis),
    row=1, col=1
)
fig.update_xaxes(
    title_text="Workload Complexity", 
    title_font=dict(size=font_size_axis),
    tickfont=dict(size=font_size_axis),
    row=1, col=1
)

# Row 2 Axis Styling
fig.update_yaxes(
    title_text="Throughput (ksps)", 
    title_font=dict(size=font_size_axis),
    tickfont=dict(size=font_size_axis),
    row=2, col=1
)
fig.update_xaxes(
    title_text="Workload Complexity", 
    title_font=dict(size=font_size_axis),
    tickfont=dict(size=font_size_axis),
    row=2, col=1
)

# Update Subplot Titles Font
fig.update_annotations(font_size=font_size_title)

fig.show()

## MindGame Dataset

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------------------------------------------------------
# 1. Load the datasets
# ---------------------------------------------------------
# Updated paths for the MindGame dataset
file_paths = [
    f'{DATA_DIR}/MindGame/MindGame_ACC_GYRO_Trace_1.csv',
    f'{DATA_DIR}/MindGame/MindGame_ACC_GYRO_Trace_2.csv',
    f'{DATA_DIR}/MindGame/MindGame_ACC_GYRO_Trace_3.csv',
    f'{DATA_DIR}/MindGame/MindGame_HR_Trace_1.csv'
]

dfs = []
for f in file_paths:
    try:
        dfs.append(pd.read_csv(f))
    except FileNotFoundError:
        print(f"Error: {f} not found.")

if not dfs:
    print("No data loaded. Please check file paths.")
    exit()

# ---------------------------------------------------------
# 2. Combine and Clean the data
# ---------------------------------------------------------
df_combined = pd.concat(dfs, ignore_index=True)

# Map the file_name to readable labels for the X-axis
name_mapping = {
    'acc.csv': 'Accelerometer',
    'gyro.csv': 'Gyroscope',
    'hr.csv': 'Heart Rate'
}
df_combined['signal_type'] = df_combined['file_name'].map(lambda x: name_mapping.get(x, x))

# 3. Aggregate: Average per Signal Type and Active Trace Count
df_agg = df_combined.groupby(['signal_type', 'active_trace_count'])[['plot_gen_time_ms', 'throughput_ksps']].mean().reset_index()

# ---------------------------------------------------------
# 4. Create Subplots (Vertical Layout for Two-Column Papers)
# ---------------------------------------------------------
# Optimized font sizes for visibility
font_size_axis = 18
font_size_title = 20
font_size_legend = 18

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("(a) <b>Average Plot Generation Time</b>", 
                    "(b) <b>Average System Throughput</b>"),
    vertical_spacing=0.15
)

# Color palette for 1, 2, and 3 active traces
colors = {1: '#EF553B', 2: '#636EFA', 3: '#00CC96'}
trace_counts = sorted(df_agg['active_trace_count'].unique())

for count in trace_counts:
    subset = df_agg[df_agg['active_trace_count'] == count]
    
    # --- Subplot 1: Plot Gen Time (Row 1) ---
    fig.add_trace(
        go.Bar(
            x=subset['signal_type'],
            y=subset['plot_gen_time_ms'],
            name=f'{count} Active Trace(s)',
            legendgroup=str(count),
            marker_color=colors.get(count, '#333'),
            offsetgroup=str(count)
        ),
        row=1, col=1
    )

    # --- Subplot 2: Throughput (Row 2) ---
    fig.add_trace(
        go.Bar(
            x=subset['signal_type'],
            y=subset['throughput_ksps'],
            name=f'{count} Active Trace(s)',
            legendgroup=str(count),
            showlegend=False,
            marker_color=colors.get(count, '#333'),
            offsetgroup=str(count)
        ),
        row=2, col=1
    )

# ---------------------------------------------------------
# 5. Global Styling & Layout update
# ---------------------------------------------------------
fig.update_layout(
    title_text="Performance Comparison: ACC, GYRO, and HR Data",
    title_x=0.5,
    title_font_size=22,
    barmode='group',
    height=1000, # Increased height for vertical stacking
    width=800,   # Width optimized for column-width scaling
    template="plotly_white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=font_size_legend)
    ),
    margin=dict(l=80, r=40, t=150, b=80)
)

# Axis formatting - Row 1
fig.update_xaxes(
    title_text="Signal Type", 
    title_font=dict(size=font_size_axis),
    tickfont=dict(size=font_size_axis),
    row=1, col=1
)
fig.update_yaxes(
    title_text="Time (ms)", 
    title_font=dict(size=font_size_axis),
    tickfont=dict(size=font_size_axis),
    row=1, col=1
)

# Axis formatting - Row 2
fig.update_xaxes(
    title_text="Signal Type", 
    title_font=dict(size=font_size_axis),
    tickfont=dict(size=font_size_axis),
    row=2, col=1
)
fig.update_yaxes(
    title_text="Throughput (ksps)", 
    title_font=dict(size=font_size_axis),
    tickfont=dict(size=font_size_axis),
    row=2, col=1
)

# Update Subplot Titles Font
fig.update_annotations(font_size=font_size_title)

fig.show()